# 02 — X-IIoTID Feature Selection and Data Quality

**Objective:** Check the 19 candidate network features and create a reduced dataset.

**Input:** Original X-IIoTID CSV and Notebook 01 summary.

**Output:** Feature-quality table, missing-value graph, reduced CSV and JSON report.

**Note:** The 19 columns are *candidates*. Final feature usefulness and edge-gateway availability still need validation.

In [ ]:
from pathlib import Path
from collections import Counter
import gzip
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
RAW = ROOT / "data/raw/x_iiotid/X-IIoTID dataset.csv"
AUDIT = ROOT / "reports/audits/x_iiotid_v3_dataset_analysis.json"
OUT = ROOT / "data/processed/x_iiotid/x_iiotid_core19_selected_v3.csv.gz"
REPORT = ROOT / "reports/audits/x_iiotid_v3_feature_quality.json"
FIG = ROOT / "reports/figures/x_iiotid_v3_feature_missingness.png"

CORE19 = [
    'Protocol', 'Service', 'Duration', 'Scr_bytes', 'Des_bytes', 'Conn_state',
    'is_syn_only', 'Is_SYN_ACK', 'is_pure_ack', 'is_with_payload', 'FIN or RST',
    'Scr_pkts', 'Scr_ip_bytes', 'Des_pkts', 'Des_ip_bytes', 'total_bytes',
    'total_packet', 'paket_rate', 'byte_rate'
]
LABELS = ['class1', 'class2', 'class3']
CATEGORICAL = ['Protocol','Service','Conn_state','is_syn_only','Is_SYN_ACK',
               'is_pure_ack','is_with_payload','FIN or RST']
NUMERIC = [c for c in CORE19 if c not in CATEGORICAL]
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown'}
CHUNK_SIZE = 30000

if not RAW.is_file() or not AUDIT.is_file():
    raise FileNotFoundError('Run Notebook 01 and check the X-IIoTID source file.')
audit = json.loads(AUDIT.read_text(encoding='utf-8'))
required = CORE19 + LABELS
actual = pd.read_csv(RAW, nrows=0).columns.tolist()
if any(c not in actual for c in required):
    raise ValueError('Dataset is missing required Core19/label columns.')
if audit['core19_features'] != CORE19 or audit['target'] != 'class3':
    raise ValueError('Core19 definition does not match Notebook 01.')
OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)
FIG.parent.mkdir(parents=True, exist_ok=True)
print('Network features:', len(CORE19), '| Additional labels:', len(LABELS))
print('Target: class3 | Class labels are never model inputs')

## 1. Check features and export the selected dataset

Count missing values, invalid numbers and repeated network-feature patterns. Keep the original values; do **not** fill missing values before splitting.

In [ ]:
missing_counts = Counter()
invalid_numeric = Counter()
missing_by_class = {c: Counter() for c in CORE19}
label_mismatches = Counter()
class_counts = Counter()
signatures = {}  # BLAKE2b digest -> [rows, normal rows, attack rows]
rows = 0
tmp = OUT.with_name(OUT.name + '.tmp')
if tmp.exists():
    raise FileExistsError(f'Previous incomplete export exists: {tmp}')

try:
    with gzip.open(tmp, 'wt', encoding='utf-8', newline='') as out_file:
        for chunk in pd.read_csv(RAW, usecols=required, dtype=str,
                                 keep_default_na=False, chunksize=CHUNK_SIZE,
                                 low_memory=False):
            # Ensure stable output column order, independent of source CSV order.
            chunk = chunk[required].copy()
            y = chunk['class3'].str.strip()
            if not y.isin(['Normal', 'Attack']).all():
                raise ValueError('Unexpected class3 label. Check dataset labels.')
            class_counts.update(y.value_counts().to_dict())

            # Normalize only a temporary copy for quality diagnostics.
            normalized = pd.DataFrame(index=chunk.index)
            for feature in CORE19:
                values = chunk[feature].str.strip()
                is_missing = values.str.casefold().isin(MISSING)
                missing_counts[feature] += int(is_missing.sum())
                for cls in ('Normal', 'Attack'):
                    missing_by_class[feature][cls] += int((is_missing & y.eq(cls)).sum())
                if feature in NUMERIC:
                    numbers = pd.to_numeric(values.mask(is_missing), errors='coerce')
                    finite = np.isfinite(numbers.to_numpy(dtype=float))
                    invalid_numeric[feature] += int((~is_missing.to_numpy() & ~finite).sum())
                normalized[feature] = values.mask(is_missing, '<MISSING>')

            # Do the auxiliary labels agree with the Normal/Attack target?
            for label in ('class1', 'class2'):
                aux_is_normal = chunk[label].str.strip().str.casefold().eq('normal')
                label_mismatches[label] += int((aux_is_normal != y.eq('Normal')).sum())

            # Identical normalized Core19 values are assigned the same digest.
            for values, cls in zip(normalized.itertuples(index=False, name=None), y):
                digest = hashlib.blake2b(
                    json.dumps(values, ensure_ascii=False, separators=(',', ':')).encode('utf-8'),
                    digest_size=16
                ).digest()
                counts = signatures.setdefault(digest, [0, 0, 0])
                counts[0] += 1
                counts[1 if cls == 'Normal' else 2] += 1

            # Export a row identifier, 19 features and 3 labels (23 columns).
            chunk.insert(0, 'csv_row_index_0based', range(rows, rows + len(chunk)))
            chunk.to_csv(out_file, index=False, header=(rows == 0))
            rows += len(chunk)
            if rows % 150000 < CHUNK_SIZE:
                print(f'Processed {rows:,} rows')

    if rows != int(audit['rows']) or class_counts != Counter(audit['label_counts']['class3']):
        raise ValueError('Record or label counts differ from Notebook 01. Export not finalized.')
    tmp.replace(OUT)
except BaseException:
    if tmp.exists():
        tmp.unlink()
    raise

repeated_groups = sum(group[0] > 1 for group in signatures.values())
repeated_rows = sum(group[0] for group in signatures.values() if group[0] > 1)
conflicting_groups = sum(group[1] > 0 and group[2] > 0 for group in signatures.values())
conflicting_rows = sum(group[0] for group in signatures.values()
                       if group[1] > 0 and group[2] > 0)
print('\nRows:', f'{rows:,}', '| Distinct normalized feature patterns:', f'{len(signatures):,}')
print('Repeated feature groups:', f'{repeated_groups:,}', '| Rows in repeated groups:', f'{repeated_rows:,}')
print('Normal/Attack conflicting groups:', f'{conflicting_groups:,}', '| Affected rows:', f'{conflicting_rows:,}')
print('Auxiliary label mismatches:', dict(label_mismatches))
print('Reduced dataset:', OUT)

## 2. Review feature quality

High missing-value percentages and conflicting labels need checking. These results describe the data; they do not prove that a feature is useful for detecting attacks.

In [ ]:
quality = pd.DataFrame([
    {
        'feature': feature,
        'type': 'categorical' if feature in CATEGORICAL else 'numeric',
        'missing_pct': round(100 * missing_counts[feature] / rows, 2),
        'invalid_numeric': invalid_numeric[feature] if feature in NUMERIC else None,
        'normal_missing_pct': round(100 * missing_by_class[feature]['Normal'] / class_counts['Normal'], 2),
        'attack_missing_pct': round(100 * missing_by_class[feature]['Attack'] / class_counts['Attack'], 2),
    }
    for feature in CORE19
]).sort_values(['missing_pct', 'feature'], ascending=[False, True])
display(quality.reset_index(drop=True))
print('\nCheck first:', quality.loc[quality.missing_pct > 0, 'feature'].head(8).tolist())

In [ ]:
plot_data = quality.head(10).sort_values('missing_pct')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(plot_data['feature'], plot_data['missing_pct'])
ax.set_title('X-IIoTID — Missing values in Core19 features')
ax.set_xlabel('Missing or placeholder values (%)')
ax.grid(axis='x', alpha=0.2)
fig.tight_layout()
fig.savefig(FIG, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)
print('Saved:', FIG)

## 3. Save the report

Notebook 03 will reuse the frozen train/validation/test split. The exported file is **not** a training dataset yet.

In [ ]:
report = {
    'dataset': 'X-IIoTID',
    'source_sha256_from_notebook_01': audit['source_sha256'],
    'rows': rows,
    'selected_features': CORE19,
    'model_target': 'class3',
    'extra_labels_not_for_training': ['class1', 'class2'],
    'exported_columns': ['csv_row_index_0based'] + CORE19 + LABELS,
    'reduced_dataset': str(OUT.relative_to(ROOT)),
    'missing_or_placeholder_counts': dict(missing_counts),
    'invalid_numeric_counts': dict(invalid_numeric),
    'missing_by_class': {k: dict(v) for k, v in missing_by_class.items()},
    'label_normal_attack_mismatches': dict(label_mismatches),
    'normalized_feature_pattern_summary': {
        'distinct_patterns': len(signatures),
        'repeated_groups': repeated_groups,
        'rows_in_repeated_groups': repeated_rows,
        'conflicting_normal_attack_groups': conflicting_groups,
        'rows_in_conflicting_groups': conflicting_rows,
        'method': 'BLAKE2b-128 of 19 stripped text fields; common missing tokens unified. Diagnostic only.'
    },
    'notes': [
        'Original CSV unchanged; selected dataset preserves values and original row order.',
        'No missing-value imputation, encoding, model fitting or train/test splitting here.',
        'Core19 feature meaning and live edge-gateway extractability remain unverified.',
        'Do not use full-dataset label/missingness diagnostics to tune a model or choose features.',
        'Model learning, supervised feature selection and preprocessing must use training data only.'
    ]
}
REPORT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved report:', REPORT)
print('Reduced dataset rows:', f'{rows:,}', '| Columns:', len(report['exported_columns']))
print('Next: 03_data_preparation.ipynb — preserve the frozen split')